In [0]:
%pip install sqlglot -q
%restart_python

In [0]:
import sqlglot
import re
from pyspark.sql.functions import udf, col, explode, split, when, size, element_at, trim, upper,expr
from pyspark.sql.types import ArrayType, StringType
from sqlglot.expressions import Table, CTE, Column
import pandas as pd
from pyspark.sql.functions import pandas_udf

In [0]:


def _strip_bo_func(sql: str, func_name: str, placeholder: str = "'PLACEHOLDER'") -> str:
    """
    Remove all occurrences of func_name(...) from sql, handling arbitrary
    parenthesis nesting depth — case-insensitive, so @Prompt/@prompt/@PROMPT
    are all matched. Replaces each occurrence with placeholder.
    """
    result    = []
    i         = 0
    fn_lower  = func_name.lower()
    fn_len    = len(func_name)
    sql_lower = sql.lower()          # used only for case-insensitive search

    while i < len(sql):
        idx = sql_lower.find(fn_lower, i)   # case-insensitive find
        if idx == -1:                        # no more occurrences
            result.append(sql[i:])
            break
        result.append(sql[i:idx])            # text before this occurrence
        paren_start = sql.find('(', idx + fn_len)
        if paren_start == -1:                # malformed: no opening paren
            result.append(sql[idx:])
            break
        # Walk forward counting depth to find the matching closing paren
        depth, j = 0, paren_start
        while j < len(sql):
            if sql[j] == '(':   depth += 1
            elif sql[j] == ')': depth -= 1
            if depth == 0:      break
            j += 1
        result.append(placeholder)
        i = j + 1                            # continue after the closing paren
    return ''.join(result)


def _strip_outer_parens(sql: str) -> str:
    """
    SAP BO wraps many queries as ( SELECT ... ) with one outer paren pair.
    Strips them only when:
      - the string starts with '('
      - the content inside starts with SELECT or WITH
      - the matching closing ')' is at the end of the string (nothing follows)
    Uses balanced-paren walking so inner subqueries are never disturbed.
    """
    s = sql.strip()
    if not s.startswith('('):
        return sql
    # Walk to find the closing paren matching the first (
    depth = 0
    close_idx = -1
    for i, ch in enumerate(s):
        if ch == '(':   depth += 1
        elif ch == ')': depth -= 1
        if depth == 0:
            close_idx = i
            break
    if close_idx == -1:
        return sql  # unbalanced — leave as-is
    inner      = s[1:close_idx].strip()
    after_paren = s[close_idx + 1:].strip().rstrip(';').strip()
    # Only strip if: inner starts with SELECT/WITH AND nothing follows closing )
    if after_paren == '' and re.match(r'^(SELECT|WITH)\b', inner, re.IGNORECASE):
        return inner
    return sql


def clean_bo_sql(sql: str) -> str:
    """Remove SAP BO-specific syntax that sqlglot cannot parse."""
    sql = _strip_bo_func(sql, "@Prompt",   placeholder="'PROMPT_PLACEHOLDER'")
    sql = _strip_bo_func(sql, "@Variable", placeholder="'VAR_PLACEHOLDER'")
    sql = _strip_bo_func(sql, "@dpvalue",  placeholder="'DPObject_PLACEHOLDER'")
    # Strip inline -- comments before parsing.
    # SAP BO stores SQL as a single line — a '--' comment would consume all
    # remaining text including closing parens, causing sqlglot parse errors.
    sql = re.sub(r'--[^\n]*', '', sql)
    # Normalise spaced comparison operators: > = → >=, < = → <=, ! = → !=
    # SAP BO sometimes emits these with a space between the two characters.
    sql = re.sub(r'>\s+=', '>=', sql)
    sql = re.sub(r'<\s+=', '<=', sql)
    sql = re.sub(r'!\s+=', '!=', sql)
    # Strip T-SQL bracket-quoted identifiers: [schema].[table] → schema.table
    # Needed for SQL Server / Azure SQL queries stored in SAP BO.
    sql = re.sub(r'\[([^\]]+)\]', r'\1', sql)
    sql = _strip_outer_parens(sql)   # handles ( SELECT ... ) SAP BO wrapper
    return sql

def extract_tables(sql):
    try:
        # Pre-process to remove SAP BO-specific syntax
        clean_sql = clean_bo_sql(sql or "")
        tables = set()
        cte_names = set()
        for parsed in sqlglot.parse(clean_sql, read="oracle", error_level=sqlglot.ErrorLevel.IGNORE):
            if parsed is None:
                continue
            # Collect CTE names to exclude them from table references
            for cte in parsed.find_all(CTE):
                cte_names.add(cte.alias.lower())
            # Collect table references
            for t in parsed.find_all(Table):
                catalog = t.catalog
                schema = t.db
                name = t.name
                full_name = ".".join(
                    part for part in [catalog, schema, name] if part
                )
                # Exclude CTE names
                if full_name.lower() not in cte_names:
                    tables.add(full_name)
        return sorted(tables)
    except Exception as e:
        print(f"Error parsing SQL: {e}")
        return []

def extract_columns(sql):
    """Extract column references from SQL, returning list of 'table.column' or 'column' strings."""
    try:
        clean_sql = clean_bo_sql(sql or "")
        columns = set()
        # ErrorLevel.WARN: sqlglot records parse issues internally instead of
        # raising — handles cases like Oracle reserved words used as aliases
        # (e.g. 'inner', 'select') without aborting the entire parse.
        for parsed in sqlglot.parse(clean_sql, read="oracle", error_level=sqlglot.ErrorLevel.IGNORE):
            if parsed is None:
                continue
            for c in parsed.find_all(Column):
                col_name = c.name
                table_ref = c.table  # table alias or name if qualified
                if col_name:
                    if table_ref:
                        columns.add(f"{table_ref}.{col_name}")
                    else:
                        columns.add(col_name)
        return sorted(columns)
    except Exception as e:
        print(f"Error parsing SQL (columns): {e}")
        return []


In [0]:


TARGET_TABLE = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_SQL_Fields"
SOURCE_TABLE = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms"

# How many Document_Ids to process before writing a checkpoint to Delta
CHECKPOINT_EVERY = 500

# ── Step 1: Incremental set difference (as per project pattern) ──────────────
try:
    _processed_pdf = spark.table(TARGET_TABLE).select("Document_Id").distinct().toPandas()
    processed_ids = set(_processed_pdf["Document_Id"].tolist())
except Exception:
    processed_ids = set()  # Target table does not exist yet

all_ids = set(
    spark.table(SOURCE_TABLE).select("Document_Id").distinct()
    .toPandas()["Document_Id"].tolist()
)
new_ids = all_ids - processed_ids
print(f"Source: {len(all_ids)} IDs | Processed: {len(processed_ids)} | New to parse: {len(new_ids)}")

if not new_ids:
    print("Nothing to process — target table is up to date.")
else:
    # ── Step 2: Collect deduplicated (Document_Id, SQL_Query) rows to driver ──
    # dropDuplicates on SQL_Query per Document_Id avoids redundant sqlglot parses.
    # With ~22K Document_Ids and ~32K distinct SQL strings this stays well under
    # driver memory limits.
    source_pdf = (
        spark.table(SOURCE_TABLE)
        .filter(
            col("Document_Id").isin(list(new_ids)) &
            col("SQL_Query").isNotNull() &
            (trim(col("SQL_Query")) != "") &
            # Exclude known non-SQL status messages from SAP BO
            ~trim(col("SQL_Query")).isin(
                "Data Source Type excel not handled for SQL extraction",
                "Error retrieving Query Plan"
            )
        )
        .select("Document_Id", "Document_name", "Connection_Name", "SQL_Query")
        .dropDuplicates(["Document_Id", "SQL_Query"])
        .toPandas()
    )
    print(f"Collected {len(source_pdf):,} distinct (Document_Id, SQL_Query) rows")

    # ── Step 3: Parse one Document_Id at a time in Python (no Spark UDF) ──────
    doc_ids = sorted(source_pdf["Document_Id"].unique())
    total   = len(doc_ids)
    buffer  = []   # accumulates parsed rows between checkpoints
    written = 0    # total rows written so far

    for idx, doc_id in enumerate(doc_ids, 1):
        doc_rows = source_pdf[source_pdf["Document_Id"] == doc_id]

        for _, row in doc_rows.iterrows():
            for col_ref in extract_columns(row["SQL_Query"]):
                parts = col_ref.split(".")
                buffer.append({
                    "Document_Id"         : str(row["Document_Id"]),
                    "Document_name"       : row["Document_name"],
                    "source_DB_connection": row["Connection_Name"],
                    "column_ref"          : col_ref,
                    "table_ref"           : parts[0].upper() if len(parts) > 1 else None,
                    "column_name"         : parts[-1].upper()
                })

        # ── Checkpoint: write buffer to Delta every CHECKPOINT_EVERY docs ──
        if idx % CHECKPOINT_EVERY == 0 or idx == total:
            if buffer:
                df_chunk = spark.createDataFrame(
                    pd.DataFrame(buffer).drop_duplicates()
                )
                df_chunk.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TARGET_TABLE)
                written += len(buffer)
                buffer = []
            pct = idx / total * 100
            print(f"  [{idx:>6}/{total}] {pct:5.1f}% complete | rows written so far: {written:,}")

    print(f"\nDone — parsed {total} Document_Id(s), wrote {written:,} rows to {TARGET_TABLE}")

# Expose full target table for downstream cells
result_columns = spark.table(TARGET_TABLE)

In [0]:

def extract_webi_headers():
    df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers")
    df_headers = df.select(
        col("Report_Type"),
        col("Report_Document_ID"),
        explode(
            split(col("Header_Content"), r"\|")
        ).alias("Header_Column")
    ).withColumn("Header_Column", trim(col("Header_Column")))
    # Replace spaces with underscores and remove non-alphanumeric/underscore characters
    from pyspark.sql.functions import regexp_replace

    df_headers = df_headers.withColumn(
        "Header_Column",
        regexp_replace(
            regexp_replace(col("Header_Column"), " ", "_"),
            r"[^A-Za-z0-9_]", ""
        )
    )
    df_headers=df_headers.dropDuplicates()
    return df_headers

# display(extract_webi_headers())

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables

In [0]:
df_header = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_output_headers_columns")
df_header = df_header.filter(col("Report_Document_ID") == 411447)
from pyspark.sql.functions import regexp_replace, lower, trim

# Normalize Header_Column: lowercase, trim, remove spaces, remove non-alphanumeric/underscore
df_header = df_header.withColumn(
    "Header_Column",
    regexp_replace(
        regexp_replace(
            lower(trim(col("Header_Column"))),
            " ", ""
        ),
        r"[^a-z0-9_]", ""
    )
).dropDuplicates(["Report_Document_ID", "Header_Column"])

print("Number of Header_Column in this document:", df_header.count())
df_variables = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables") \
    .select(
        col("Document_Id"),
        col("variable_name")
    )
# Add a new column: match from result_columns (column_name == Header_Column) with upper(trim())
df_result_columns = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_SQL_Fields") \
    .select(
        col("Document_Id").alias("rc_Document_Id"),
        col("column_ref").alias("rc_column_ref"),
        col("column_name").alias("rc_column_name")
    )

# Use ai_similarity to semantically match Header_Column to variable_name and rc_column_name

# Compute similarity scores
df_llm_input = df_header.join(
    df_variables,
    df_header.Report_Document_ID == df_variables.Document_Id,
    how="left"
).select(
    df_header.Report_Document_ID,
    df_header.Header_Column,
    df_variables.variable_name,
    # df_result_columns.rc_column_ref
).dropDuplicates()
df_llm_input2 = df_header.join(
    df_result_columns,
    df_header.Report_Document_ID == df_result_columns.rc_Document_Id,
    how="left"
).select(
    df_header.Report_Document_ID,
    df_header.Header_Column,
    df_result_columns.rc_column_ref
).dropDuplicates()


# Compute similarity scores for header-variable and header-result_column
df_sim = df_llm_input.withColumn(
    "sim_variable",
    expr("ai_similarity(Header_Column, variable_name)")
)
df_sim2 = df_llm_input2.withColumn(
    "sim_result_col",
    expr("ai_similarity(Header_Column, rc_column_ref)")
)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

threshold = 0.75

# For each header, find best variable match above threshold
w_var = Window.partitionBy("Report_Document_ID", "Header_Column").orderBy(col("sim_variable").desc_nulls_last())
df_var_match = df_sim.filter(col("sim_variable") >= threshold) \
    .withColumn("rn_var", row_number().over(w_var)) \
    .filter(col("rn_var") == 1) \
    .select(
        "Report_Document_ID",
        "Header_Column",
        col("variable_name").alias("OutPut_matched"),
        col("sim_variable").alias("similarity_score"),
        expr("'variable'").alias("match_type")
    )
df_var_match.createOrReplaceTempView("temp_var_match")
display(df_var_match)
w_rc = Window.partitionBy("Report_Document_ID", "Header_Column").orderBy(col("sim_result_col").desc_nulls_last())
df_rc_match = df_sim2.filter(col("sim_result_col") >= threshold) \
    .withColumn("rn_rc", row_number().over(w_rc)) \
    .filter(col("rn_rc") == 1) \
    .select(
        "Report_Document_ID",
        "Header_Column",
        col("rc_column_ref").alias("OutPut_matched"),
        col("sim_result_col").alias("similarity_score"),
        expr("'SQL_colum'").alias("match_type")
    )
display(df_rc_match)
from pyspark.sql.functions import col, max_by

# Union variable and result column matches
df_all_matches = df_var_match.unionByName(df_rc_match)

from pyspark.sql.functions import row_number, lit
from pyspark.sql.window import Window

# For each (Report_Document_ID, Header_Column), select the row with highest similarity_score
w_best = Window.partitionBy("Report_Document_ID", "Header_Column").orderBy(col("similarity_score").desc_nulls_last())
df_best_match = df_all_matches.withColumn(
    "rn", row_number().over(w_best)
).filter(col("rn") == 1).drop("rn")

# Find all (Report_Document_ID, Header_Column) that have no match above threshold
df_all_headers = df_header.select("Report_Document_ID", "Header_Column").dropDuplicates()
df_missing = df_all_headers.join(
    df_best_match,
    on=["Report_Document_ID", "Header_Column"],
    how="left_anti"
).withColumn("OutPut_matched", lit(None)) \
 .withColumn("similarity_score", lit(None)) \
 .withColumn("match_type", lit(None))

df_final = df_best_match.unionByName(df_missing)

display(df_final)
# # Union variable matches and rc_column_name matches
# df_final = df_var_match.unionByName(df_rc_match)

# display(df_final)